# EdgeDeploy: ONNX Runtime Benchmark & Quantization

**arXiv ref:** *Selective Quantization Tuning for ONNX Models* — arxiv:2507.12196

**Edge AI Benchmark:** github.com/joeltadeu/edge-ai-benchmark

This notebook demonstrates MobileNet-V2 deployment with ONNX export, INT8 quantization, and inference benchmarking.


In [ ]:
from __future__ import annotations
import sys, os; sys.path.insert(0, os.path.abspath('.'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import make_synthetic, create_dataloaders
from src.model import build_mobilenet, evaluate_model, export_to_onnx, benchmark_onnx, quantize_onnx, get_model_size


In [ ]:
# Generate evaluation data
data = make_synthetic(n=200, seed=42)
_, val_loader = create_dataloaders(data, batch_size=32, seed=42)
print(f"Validation samples: {len(val_loader.dataset)}")


In [ ]:
# Build MobileNet-V2
model = build_mobilenet(num_classes=1000)
print(f"MobileNet-V2: {sum(p.numel() for p in model.parameters()):,} params")


In [ ]:
# Benchmark PyTorch FP32
pt_metrics = evaluate_model(model, val_loader)
print(f"PyTorch FP32: {pt_metrics['avg_latency_ms']:.2f}ms, {pt_metrics['throughput_fps']:.0f} fps")


In [ ]:
# Export to ONNX
onnx_path = export_to_onnx(model)
print(f"ONNX model size: {get_model_size(onnx_path):.2f} MB")


In [ ]:
# Benchmark ONNX FP32
onnx_metrics = benchmark_onnx(onnx_path, val_loader)
print(f"ONNX FP32: {onnx_metrics['avg_latency_ms']:.2f}ms, {onnx_metrics['throughput_fps']:.0f} fps")


In [ ]:
# Quantize to INT8
quant_path = quantize_onnx(onnx_path)
quant_size = get_model_size(quant_path)
print(f"Quantized size: {quant_size:.2f} MB ({(1 - quant_size / get_model_size(onnx_path)) * 100:.0f}% reduction)")


In [ ]:
# Benchmark ONNX INT8
quant_metrics = benchmark_onnx(quant_path, val_loader)
print(f"ONNX INT8: {quant_metrics['avg_latency_ms']:.2f}ms, {quant_metrics['throughput_fps']:.0f} fps")


In [ ]:
print('EdgeDeploy notebook complete. ONNX INT8 quantization reduces model size by ~75% with minimal accuracy loss.')
